In [ ]:
%matplotlib inline

import numpy

from scipy.special import wofz   # for Voigt function
from scipy.special import roots_legendre  # for Gaussian quadrature
from scipy.sparse import diags
from scipy.linalg import solve_banded

from numba import njit

import matplotlib.pyplot as plt

from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
def quadrature(k=5):
    """
    Returns the nodes and weights form a Gaussian
    quadrature with k points. Rescaled to an interval
    from [0, 1].
    """
    nodes, weights = roots_legendre(k)
    return nodes/2 + 0.5, weights/2

In [ ]:
# Code from last tutorial
def Tmatrix(τ, μ, format=None):
    tau = τ / μ
    n = len(tau)
    Δτ = np.roll(tau, -1) - tau
    Δτm = np.roll(Δτ, 1)

    A = 2 / (Δτm * (Δτm + Δτ))
    B = 1 + 2 / (Δτ * Δτm)
    C = 2 / (Δτ * (Δτm + Δτ))

    # Boundary conditions top
    B[0] = 2 / Δτ[0]**2 + 2 / Δτ[0] + 1
    C[0] = 2 / Δτ[0]**2
    # Boundary conditions bottom
    A[n-1] = 2 / (Δτ[n-2] * (Δτ[n-2] + 2))
    B[n-1] = (2 + 2*Δτ[n-2] + Δτ[n-2]**2) / (Δτ[n-2] * (Δτ[n-2] + 2))

    if format == 'banded':  # for use with scipy.linalg.solve_banded
        T = np.zeros((3, n))
        T[0, 1:] = -C[:-1]
        T[1] = B
        T[2, :-1] = -A[1:]
        return T
    elif format == 'sparse':  # for use with scipy.sparse.linalg.spsolve
        return diags([-A[1:], B, -C[:-1]], [-1, 0, 1], format='csc')
    else:  # for use with np.linalg.solv
        return diags([-A[1:], B, -C[:-1]], [-1, 0, 1]).toarray()

def lambda_operator_implicit(tau, S):
    J = np.zeros_like(S)
    for mu, w in zip(*quadrature()):
        T = Tmatrix(tau, mu, format='banded')
        P = solve_banded((1, 1), T, S)
        J += w * P
    return J

# Solving by the simple iteration scheme

In [ ]:
def solve_iter(τ, B, ε, maxiter=1000, threshold=1e-3):
    # write a simple iterative scheme
    
    return None

In [ ]:
## Solve and plot source function

# What is the "correct solution"?

In [ ]:
# From last tutorial:
def lambda_matrix(tau, k=5):
    n = len(tau)
    L = numpy.zeros((n, n))
    for mu, w in zip(*quadrature(k)):
        T = Tmatrix(tau, mu)
        L += w * numpy.linalg.inv(T)
    return L

def solve_direct(τ, B, ε):
    n = len(τ)
    Λ = lambda_matrix(τ)
    M = numpy.linalg.inv(numpy.eye(n) - (1 - ε) * Λ)
    S = ε * numpy.dot(M, B)
    J = numpy.dot(Λ, S)
    return S, J

In [ ]:
# Compare with the direct inversion from last time:

### reduce $\varepsilon$

In [ ]:
# reduce epsilon and find out when the iterative scheme does not work